<h1 style="color:#C0392B; font-family:Georgia,serif; text-align:center; padding:20px 0; border-bottom:3px solid #C0392B;">
Text Summarization: Extractive and Abstractive Approaches
</h1>

<p style="text-align:center; font-family:Georgia,serif; font-size:15px; color:#555;">
A structured pipeline combining TF-IDF extractive summarization and BART-based abstractive summarization.
</p>

---
<h2 style="color:#C0392B; font-family:Georgia,serif; background:#fdf2f2; padding:12px 18px; border-left:6px solid #C0392B; border-radius:4px;">
Part 1 — Setup and Preprocessing
</h2>

This section installs all required dependencies, imports libraries, loads the dataset, and applies preprocessing steps necessary for both the TF-IDF and BART pipelines.

<h3 style="color:#C0392B; font-family:Georgia,serif;">1.0 — Dependency Installation</h3>

PyTorch must be installed **before** importing `transformers`. The CPU-only build is used here for compatibility across all environments.

> **Required workflow:** Run this cell once → wait for it to complete → **Kernel → Restart** → then run all remaining cells in order.

In [ ]:
# ── Run this cell once, then restart the kernel before running anything else ──
import subprocess, sys

# Step 1: Install CPU-only PyTorch (must come before transformers)
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cpu",
    "--quiet"
])

# Step 2: Install remaining dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "transformers", "scikit-learn", "nltk", "pandas", "numpy",
    "--quiet"
])

print("Installation complete.")
print("ACTION REQUIRED: Go to Kernel > Restart, then run all remaining cells.")

CalledProcessError: Command '['c:\\Users\\e\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', '-m', 'pip', 'install', 'torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cpu', '--quiet']' returned non-zero exit status 1.

<h3 style="color:#C0392B; font-family:Georgia,serif;">1.1 — Environment Verification</h3>

Confirm that PyTorch is available before importing the `transformers` library. If this cell raises an error, re-run cell 1.0 and restart the kernel.

In [2]:
try:
    import torch
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}  (False is expected on CPU-only builds)")
except ImportError:
    raise ImportError(
        "PyTorch not found. Run cell 1.0 first, restart the kernel, then re-run from this cell."
    )

PyTorch version : 2.11.0+cpu
CUDA available  : False  (False is expected on CPU-only builds)


<h3 style="color:#C0392B; font-family:Georgia,serif;">1.2 — Imports</h3>

In [3]:
import re
import warnings
import torch
import nltk
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from transformers import BartTokenizer, BartForConditionalGeneration

warnings.filterwarnings('ignore')

# Download required NLTK resources if not already present
for pkg, path in [("punkt",     "tokenizers/punkt"),
                  ("punkt_tab", "tokenizers/punkt_tab"),
                  ("stopwords", "corpora/stopwords")]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(pkg, quiet=True)

STOP_WORDS = set(stopwords.words('english'))
print("All libraries imported successfully.")

C:\Users\Menna\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Menna\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


All libraries imported successfully.


<h3 style="color:#C0392B; font-family:Georgia,serif;">1.3 — Data Loading</h3>

Load a sample of 50,000 records to ensure manageable memory usage during development and evaluation.

In [4]:
SAMPLE_SIZE = 50_000
DATA_PATH   = 'data.csv'

raw_df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE)
df     = raw_df[['Content', 'Summary']].copy()

print(f"Loaded  : {len(df):,} records")
print(f"Columns : {df.columns.tolist()}")
df.head(3)

Loaded  : 50,000 records
Columns : ['Content', 'Summary']


,Content,Summary
0,New York police are concerned drones could bec...,Police have investigated criminals who have ri...
1,By . Ryan Lipman . Perhaps Australian porn sta...,Porn star Angela White secretly filmed sex act...
2,"This was, Sergio Garcia conceded, much like be...",American draws inspiration from fellow country...


<h3 style="color:#C0392B; font-family:Georgia,serif;">1.4 — Data Cleaning</h3>

Remove null values, duplicate rows, and samples below the minimum length threshold.

In [5]:
MIN_CONTENT_LENGTH = 100
initial_size       = len(df)

df.dropna(subset=['Content', 'Summary'], inplace=True)
df.drop_duplicates(subset=['Content'], inplace=True)
df = df[df['Content'].str.len() >= MIN_CONTENT_LENGTH]
df.reset_index(drop=True, inplace=True)

print(f"Removed  : {initial_size - len(df):,} invalid records")
print(f"Retained : {len(df):,} clean records")

Removed  : 1,070 invalid records
Retained : 48,930 clean records


<h3 style="color:#C0392B; font-family:Georgia,serif;">1.5 — Text Preprocessing</h3>

Normalize the `Content` column for TF-IDF processing. The original `Content` is preserved separately — BART performs best on unmodified natural-language text.

In [6]:
def clean_text(text: str) -> str:
    """Normalize text for TF-IDF processing."""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)            # remove punctuation and digits
    text = re.sub(r'\s+', ' ', text).strip()        # collapse whitespace
    return text


df['Content_Clean'] = df['Content'].apply(clean_text)

print("Original :", df['Content'].iloc[0][:120])
print("Cleaned  :", df['Content_Clean'].iloc[0][:120])

Original : New York police are concerned drones could become tools for terrorists, and are investigating ways to stop potential att
Cleaned  : new york police are concerned drones could become tools for terrorists and are investigating ways to stop potential atta


<h3 style="color:#C0392B; font-family:Georgia,serif;">1.6 — Train / Test Split</h3>

Partition into training (80%) and test (20%) subsets with a fixed random seed for reproducibility.

In [7]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

print(f"Training samples : {len(train_df):,}")
print(f"Test samples     : {len(test_df):,}")

Training samples : 39,144
Test samples     : 9,786


---
<h2 style="color:#C0392B; font-family:Georgia,serif; background:#fdf2f2; padding:12px 18px; border-left:6px solid #C0392B; border-radius:4px;">
Part 2 — TF-IDF Extractive Summarization
</h2>

Extractive summarization selects the most informative sentences from the source document. Sentence importance is scored using TF-IDF weights and the top-N sentences are returned in original document order to preserve coherence.

In [8]:
def tfidf_summary(text: str, n_sentences: int = 3) -> str:
    """
    Generate an extractive summary using TF-IDF sentence ranking.

    Parameters
    ----------
    text        : str  — Input document text.
    n_sentences : int  — Number of top sentences to include.

    Returns
    -------
    str — Extracted summary in original sentence order.
    """
    sentences = sent_tokenize(text)

    if len(sentences) <= n_sentences:
        return text

    vectorizer      = TfidfVectorizer(stop_words='english')
    tfidf_matrix    = vectorizer.fit_transform(sentences)
    sentence_scores = np.array(tfidf_matrix.mean(axis=1)).flatten()
    top_indices     = sorted(np.argsort(sentence_scores)[-n_sentences:])

    return ' '.join([sentences[i] for i in top_indices])

In [9]:
# Demonstrate on one test sample
sample_text  = test_df['Content'].iloc[0]
tfidf_result = tfidf_summary(sample_text, n_sentences=3)

print("=== Original Text (first 400 chars) ===")
print(sample_text[:400])
print()
print("=== TF-IDF Extractive Summary ===")
print(tfidf_result)

=== Original Text (first 400 chars) ===
After a rain-delayed start, an innings-best partnership of 91 between Will Smith (93) and Gareth Berg (50) carried the hosts' score past 400.
Neil Dexter claimed four of the five wickets to fall in the day with his medium pace to achieve figures of 5-64.
Joe Burns (38) edged the penultimate ball of the day behind to leave Middlesex 102-3, still 311 runs behind.
Rain meant that play got under way a

=== TF-IDF Extractive Summary ===
Rain meant that play got under way at 14:15 BST and Hampshire lost Sean Ervine early on when he was bowled by Toby Roland-Jones after adding just one run. Hampshire began brightly with the ball and debutant Brad Wheal produced a quick delivery which nipped back to bowl Sam Robson, before Edwards struck with his first delivery to pin Nick Gubbins lbw. The away side appeared to have weathered the early pressure but with the partnership worth 69 runs and Nick Compton 32 not out, Burns edged Ervine to Adam Wheater to under

In [10]:
# Apply TF-IDF summarization across the full test set
test_df['TF-IDF_Summary'] = test_df['Content'].apply(
    lambda x: tfidf_summary(x, n_sentences=3)
)
print(f"TF-IDF summaries generated for {len(test_df):,} test samples.")

TF-IDF summaries generated for 9,786 test samples.


---
<h2 style="color:#C0392B; font-family:Georgia,serif; background:#fdf2f2; padding:12px 18px; border-left:6px solid #C0392B; border-radius:4px;">
Part 3 — BART Abstractive Summarization
</h2>

Abstractive summarization generates novel text that captures the essence of the source document. The `facebook/bart-large-cnn` model is used — a transformer fine-tuned specifically on news summarization. Raw unmodified text is passed as input, as the model was trained on natural-language prose.

In [11]:
BART_MODEL_NAME = 'facebook/bart-large-cnn'

print(f"Loading model: {BART_MODEL_NAME} ...")
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL_NAME)
bart_model     = BartForConditionalGeneration.from_pretrained(BART_MODEL_NAME)
bart_model.eval()   # disable dropout for deterministic inference
print("Model loaded successfully.")

Loading model: facebook/bart-large-cnn ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully.


In [12]:
def bart_summary(
    text: str,
    max_input_length:  int = 1024,
    max_output_length: int = 150,
    min_output_length: int = 40,
    num_beams:         int = 4
) -> str:
    """
    Generate an abstractive summary using the BART-large-CNN model.

    Parameters
    ----------
    text              : str — Input document (unmodified natural language).
    max_input_length  : int — Maximum encoder token length.
    max_output_length : int — Maximum tokens in the generated summary.
    min_output_length : int — Minimum tokens in the generated summary.
    num_beams         : int — Beam search width.

    Returns
    -------
    str — Abstractive summary produced by BART.
    """
    inputs = bart_tokenizer(
        text,
        return_tensors='pt',
        max_length=max_input_length,
        truncation=True
    )

    with torch.no_grad():   # disable gradient tracking for inference efficiency
        summary_ids = bart_model.generate(
            inputs['input_ids'],
            max_length=max_output_length,
            min_length=min_output_length,
            num_beams=num_beams,
            length_penalty=2.0,
            early_stopping=True
        )

    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [13]:
# Demonstrate on one test sample
bart_result = bart_summary(sample_text)

print("=== Original Text (first 400 chars) ===")
print(sample_text[:400])
print()
print("=== BART Abstractive Summary ===")
print(bart_result)

=== Original Text (first 400 chars) ===
After a rain-delayed start, an innings-best partnership of 91 between Will Smith (93) and Gareth Berg (50) carried the hosts' score past 400.
Neil Dexter claimed four of the five wickets to fall in the day with his medium pace to achieve figures of 5-64.
Joe Burns (38) edged the penultimate ball of the day behind to leave Middlesex 102-3, still 311 runs behind.
Rain meant that play got under way a

=== BART Abstractive Summary ===
Neil Dexter claimed four of the five wickets to fall in the day with his medium pace. Will Smith (93) and Gareth Berg (50) carried the hosts' score past 400. Joe Burns (38) edged the penultimate ball of the day behind to leave Middlesex 102-3.


In [14]:
# Apply BART to a limited evaluation subset (constrained by CPU inference time)
EVAL_SAMPLE_SIZE        = 100
eval_df                 = test_df.head(EVAL_SAMPLE_SIZE).copy()
eval_df['BART_Summary'] = eval_df['Content'].apply(bart_summary)

print(f"BART summaries generated for {EVAL_SAMPLE_SIZE} evaluation samples.")

BART summaries generated for 100 evaluation samples.


---
<h2 style="color:#C0392B; font-family:Georgia,serif; background:#fdf2f2; padding:12px 18px; border-left:6px solid #C0392B; border-radius:4px;">
Part 4 — Pipeline Integration
</h2>

A unified `summarize_text` function wraps both approaches behind a single interface. The caller selects the desired method via the `method` parameter, enabling seamless switching between extractive and abstractive modes.

In [15]:
def summarize_text(text: str, method: str = 'bart') -> str:
    """
    Unified summarization interface.

    Parameters
    ----------
    text   : str — Input document text.
    method : str — 'tfidf' for extractive, 'bart' for abstractive.

    Returns
    -------
    str — Generated summary.

    Raises
    ------
    ValueError — If input is empty or method is unsupported.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string.")

    method = method.lower().strip()

    if method == 'tfidf':
        return tfidf_summary(text, n_sentences=3)
    elif method == 'bart':
        return bart_summary(text)
    else:
        raise ValueError(f"Unknown method '{method}'. Choose 'tfidf' or 'bart'.")

In [ ]:
# Validate both methods through the unified pipeline
test_input = test_df['Content'].iloc[1]

print(" Original Text: ")
print(test_input)

print()
print(" TF-IDF via Pipeline:")
print(summarize_text(test_input, method='tfidf'))

print()
print(" BART via Pipeline:")
print(summarize_text(test_input, method='bart'))-

 Original Text: 
By . Sam Webb for MailOnline . A man who paid a drug addict just £1,000 to kill his wife because he was having an affair with her sister was convicted has been convicted of murder. Mother-of-two Amina Bibi was stabbed at least 70 times in the attack at the couple's flat in Forest Gate, east London. Her 11-year-old son found her dying in a pool of blood when he returned to his home after forgetting his homework on the morning of Friday September 13. Husband Mohamed Ali, 65, was heavily in debt and having an affair with his sister-in-law in Pakistan when he approached drug addict Frederick Best, 47. Scroll down for video . Mother-of-two Amina Bibi was stabbed at least 70 times by an addict paid . by her husband Mohamed Ali (right). Mrs Bibi had savings in the house and officers believe the debt and his affair could have been a motive . Mrs Bibi had significant savings in the house and officers believe the debt and affair could have been a motive for Ali to have his wife 